# ImMimic UMI pick-place: dataset EDA

Inspect the two HDF5 files (robot `umi_pick_place_0_5.hdf5`, human `umi_pick_place_200_0.hdf5`): demo keys, `mask/train` / `mask/valid` splits, and per-demo contents. Then check whether a human→robot DTW pairing is already shipped.

In [1]:
import json
from pathlib import Path

import h5py
import numpy as np

DATA_DIR = Path("ImMimic-CoRL2025/robomimic/robomimic/datasets")
FILES = {
    "robot": DATA_DIR / "umi_pick_place_0_5.hdf5",
    "human": DATA_DIR / "umi_pick_place_200_0.hdf5",
}

## 1. File layout and demo keys

In [2]:
def describe(name, path):
    with h5py.File(path, "r") as f:
        data = f["data"]
        demos = sorted(data.keys(), key=lambda k: int(k.split("_")[1]))
        print(f"== {name}: {path.name}")
        print("top-level keys :", list(f.keys()))
        print("data attrs     :", {k: data.attrs[k] for k in data.attrs})
        print(f"#demos         : {len(demos)}  ({demos[0]} .. {demos[-1]})")
        print(f"{demos[0]} contents:")
        data[demos[0]].visititems(
            lambda n, o: print(f"   {n:28s} {o.shape} {o.dtype}") if isinstance(o, h5py.Dataset) else None
        )
        lens = np.array([data[d].attrs["num_samples"] for d in demos])
        print(f"length (frames): min={lens.min()} mean={lens.mean():.1f} max={lens.max()} total={lens.sum()}")
        print()
        return demos, lens

info = {name: describe(name, p) for name, p in FILES.items()}

== robot: umi_pick_place_0_5.hdf5
top-level keys : ['data', 'mask']
data attrs     : {'env_args': '{"env_name": "PickPlaceCan", "type": "1", "env_kwargs": {"type": "OSC_POSE", "contro_freq": 30}}', 'total': 5}
#demos         : 5  (demo_0 .. demo_4)
demo_0 contents:
   action_absolute              (390, 7) float64
   obs/agentview_image          (390, 180, 320, 3) uint8
   obs/eef_pose_w_gripper       (390, 8) float64
   obs/wrist_image              (390, 180, 320, 3) uint8
length (frames): min=390 mean=417.6 max=437 total=2088

== human: umi_pick_place_200_0.hdf5
top-level keys : ['data', 'mask']
data attrs     : {'env_args': '{"env_name": "PickPlaceCan", "type": "1", "env_kwargs": {"type": "OSC_POSE", "contro_freq": 30}}', 'total': 200}
#demos         : 200  (demo_0 .. demo_199)
demo_0 contents:
   action_absolute              (71, 7) float64
   obs/agentview_image          (71, 180, 320, 3) uint8
   obs/eef_pose_w_gripper       (71, 8) float64
length (frames): min=56 mean=79.9 max=11

## 2. `mask/train` and `mask/valid` splits

In [3]:
def splits(path):
    with h5py.File(path, "r") as f:
        return {k: sorted((x.decode() for x in f["mask"][k][:]), key=lambda s: int(s.split("_")[1]))
                for k in f["mask"].keys()}

masks = {name: splits(p) for name, p in FILES.items()}
for name, m in masks.items():
    demos = set(info[name][0])
    print(f"== {name}")
    for split in ("train", "valid"):
        keys = m.get(split)
        if keys is None:
            print(f"  mask/{split}: <missing>")
        else:
            print(f"  mask/{split}: {len(keys)} demos  e.g. {keys[:5]}")
    covered = set().union(*map(set, m.values()))
    print(f"  demos not in any mask: {len(demos - covered)}")

== robot
  mask/train: 5 demos  e.g. ['demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4']
  mask/valid: 0 demos  e.g. []
  demos not in any mask: 0
== human
  mask/train: 200 demos  e.g. ['demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4']
  mask/valid: <missing>
  demos not in any mask: 0


## 3. Human vs. robot demo counts

In [4]:
for name in FILES:
    demos, lens = info[name]
    n_train = len(masks[name].get("train", []))
    n_valid = len(masks[name].get("valid", []))
    print(f"{name:5s}: {len(demos):3d} demos | train={n_train} valid={n_valid} | {lens.sum()} frames")

robot:   5 demos | train=5 valid=0 | 2088 frames
human: 200 demos | train=200 valid=0 | 15976 frames


**Summary**

| | file | demos | `mask/train` | `mask/valid` | obs keys |
|---|---|---|---|---|---|
| **Robot** | `umi_pick_place_0_5.hdf5` | **5** | 5 | 0 (empty) | `agentview_image`, `wrist_image`, `eef_pose_w_gripper` |
| **Human** | `umi_pick_place_200_0.hdf5` | **200** | 200 | — (key absent) | `agentview_image`, `eef_pose_w_gripper` |

* File naming is `umi_pick_place_<#human>_<#robot>`: 0 human + 5 robot, and 200 human + 0 robot.
* Both store `action_absolute` (T, 7) = eef xyz + rotation (3) + gripper, at 30 Hz (`env_args`). Human actions are retargeted hand trajectories in the same robot action space.
* There is no validation split, so all demos are train-only. The config [`umi_pick_place_100_5.json`](ImMimic-CoRL2025/robomimic/robomimic/exps/configs/umi_pick_place_100_5.json) uses `num_human_demo: 100`, so only the first 100 of the 200 human demos are used in that run.
* Human demos are much shorter (56–113 frames, mean 80) than robot demos (390–437 frames, mean 418).

## 4. Is the human↔robot DTW pairing already provided?

Grepping configs/dataset code for `mixup|dtw|pair` finds:
* `train.mapping` in the config → `datasets/umi_pick_place_action_mapping.json`, plus `train.mixup: true`.
* `SequenceDataset.__init__` loads it (`utils/dataset.py`), and `__getitem__` samples `meta["pair_index"] = (robot_demo, robot_t)` from `mapping[human_demo][human_t]`, keeping only robot demos `< num_robot_demo_for_mixup`.
* `run_epoch` (`utils/train_utils.py`) fetches that robot sample, and `DiffusionPolicyUNet.train_on_batch` mixes obs features and actions with `alpha`.

So the pairing **is shipped precomputed**. Its structure:

In [5]:
mapping = json.load(open(DATA_DIR / "umi_pick_place_action_mapping.json"))
human_demos, human_lens = info["human"]
robot_demos, robot_lens = info["robot"]

print("format: mapping[human_demo][human_t][robot_demo] = robot_t")
print("human demos in mapping:", len(mapping))
robot_ids = sorted({int(r) for d in mapping.values() for t in d.values() for r in t})
print("robot demo ids referenced:", robot_ids[0], "..", robot_ids[-1], f"({len(robot_ids)} demos)")

# every human frame has an entry
ok = all(len(mapping[d.split("_")[1]]) == n for d, n in zip(human_demos, human_lens))
print("one entry per human frame for every human demo:", ok)

# robot indices fit in the robot demos we actually have
avail = {int(d.split("_")[1]): n for d, n in zip(robot_demos, robot_lens)}
bad = [(h, t, r, j) for h, d in mapping.items() for t, rs in d.items()
       for r, j in rs.items() if int(r) in avail and not 0 <= j < avail[int(r)]]
print("out-of-range robot indices for available robot demos:", len(bad))
print("example mapping['0']['10']:", mapping["0"]["10"])

format: mapping[human_demo][human_t][robot_demo] = robot_t
human demos in mapping: 200
robot demo ids referenced: 0 .. 20 (21 demos)
one entry per human frame for every human demo: True
out-of-range robot indices for available robot demos: 0
example mapping['0']['10']: {'0': 18, '1': 10, '10': 31, '11': 10, '12': 44, '13': 10, '14': 20, '15': 10, '16': 7, '17': 17, '18': 43, '19': 28, '2': 30, '20': 46, '3': 13, '4': 10, '5': 10, '6': 10, '7': 40, '8': 49, '9': 22}


In [6]:
# DTW-style sanity: for a given (human, robot) pair, robot_t should be monotone non-decreasing in human_t
def path(h, r):
    d = mapping[str(h)]
    return np.array([d[str(t)][str(r)] for t in range(len(d))])

mono = np.mean([np.all(np.diff(path(h, r)) >= 0) for h in range(len(mapping)) for r in avail])
print(f"fraction of (human, robot) pairs with monotone robot index: {mono:.3f}")
print("human 0 -> robot 0 path:", path(0, 0))

fraction of (human, robot) pairs with monotone robot index: 1.000
human 0 -> robot 0 path: [  0   1   2   3   4   9  10  13  15  16  18  19  23  24  25  29  30  32
  38  39 115 135 169 173 175 176 198 199 200 201 202 203 204 205 206 208
 210 215 217 222 229 232 241 248 254 255 256 257 258 259 260 261 262 263
 264 265 266 267 268 269 270 271 277 325 358 359 360 363 372 388 389]


**Takeaways**

* The pairing already exists, so you don't need to recompute DTW to train with `mixup: true`.
* The mapping refers to **21 robot demos (0–20)**, but the robot file only has 5. The loader filters these with `int(k) < num_robot_demo_for_mixup` (set to `trainset.num_demo` in `train.py`), so only robot demos 0–4 are used.
* Each human frame is mapped to one frame in *every* robot demo. During training, one of these is picked at random per sample.